# Domain-adaptive MLM training

This notebook trains the two encoder checkpoints used by the project. Run it from the `notebooks/` directory after materializing the public corpus:

```bash
python tools/materialize_datasets.py --skip-synthetic
```

Generated checkpoints are written to the ignored `artifacts/models/` directory.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings import EmbeddingModel

DATA_DIRECTORY = PROJECT_ROOT / "data" / "medicamentos" / "txt"
TEST_CSV = PROJECT_ROOT / "data" / "splits" / "test.csv"
MODEL_DIRECTORY = PROJECT_ROOT / "artifacts" / "models"
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

## BERTIN domain adaptation

Adapt the general Spanish BERTIN checkpoint to the section-4.8 corpus.

In [ ]:
bertin = EmbeddingModel(
    model_name="bertin-project/bertin-base-gaussian-exp-512seqlen",
    data_set_path=str(DATA_DIRECTORY),
    output_model_dir=str(MODEL_DIRECTORY / "CIMA-BERTIN-4.8"),
    test_training_path=str(TEST_CSV),
)

bertin.load_data_set()
print(f"Training tokens: {bertin.calculate_training_tokens():,}")
bertin.train_model()
bertin.save_model_information()

## Biomedical RoBERTa domain adaptation

Adapt the biomedical Spanish RoBERTa checkpoint with the same corpus and holdout split.

In [ ]:
roberta = EmbeddingModel(
    model_name="PlanTL-GOB-ES/roberta-base-biomedical-es",
    data_set_path=str(DATA_DIRECTORY),
    output_model_dir=str(MODEL_DIRECTORY / "CIMA-RoBERTa-4.8"),
    test_training_path=str(TEST_CSV),
)

roberta.load_data_set()
print(f"Training tokens: {roberta.calculate_training_tokens():,}")
roberta.train_model()
roberta.save_model_information()

## Evaluation

Evaluate the generated checkpoints with `run_mlm_test.py`, passing their paths with `--bertin-model` and `--roberta-model`.